# Tikhonov regularization with KL data fidelity term
We consider the two-dimensional deconvolution problems to find a non-negative function f given data 
$$
    g^{\mathrm{obs}} \sim \mathrm{Pois}(h*f)
$$
with a non-negative convolution kernel $h$, and $\mathrm{Pois}$ denotes the element-wise Poisson distribution.


In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mplib

from regpy.operators.convolution import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import Setting
from regpy.solvers.linear.semismoothNewton import SemismoothNewton_nonneg
from regpy.solvers.linear.proximal_gradient import ForwardBackwardSplitting, FISTA
from regpy.solvers.linear.primal_dual import PDHG
from regpy.hilbert import L2
from regpy.stoprules import DualityGapStopping
from regpy.functionals import QuadraticLowerBound, QuadraticBilateralConstraints, KullbackLeibler, RelativeEntropy

from plots import comparison_plot, convergence_plot, plot_recos
from test_images import mixed

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

### creating Poisson distributed synthetic data

In [ ]:
grid, exact_sol = mixed(M=256,N=256,c_bubbles=1.,fac=200.)
r"""grid is the underlying UniformGridFcts vector space, and exact_sol the exact solution."""
a=0.05
conv =  GaussianBlur(grid,a,pad_amount=16)
r"""Convolution operator $f\mapsto h*f$ for the convolution kernel $h(x)=\exp(-|x|_2^2/a^2)$."""
blur = conv(exact_sol)
blur[blur<0] = 0.
"""Simulated exact data."""
data = np.random.poisson(blur)
"""Simulated measured data. The Poisson distribution occurs if photon count detectors are used."""
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

## Kullback-Leibler data fidelity with nonnegativity-contrained $L^2$ penalty

We with to use the Kullback-Leibler divergence as data fidelity functional as it is the negative log-likelihood 
of Poisson data. Moreover, we choose the squared $L^2$-norm with a non-negativity constraint as penalty term. 
The constraint ensures nonnegativity of exact and predicted data (as convolution with the Gauss-kernel preserves nonnegativity), and hence the data fidelity term is always finite. This leads to the minimization problem
$$
\hat{f} \in \mathrm{argmin}_{f\geq 0}\left[\frac{1}{\alpha}\mathrm{KL}(g^{\mathrm{obs}},Tf)+\|f\|^2\right].   
$$ 
Let us look at the available methods for solving this minimization problem:

In [ ]:
alpha = 1e-3 # regularization parameter
KL = KullbackLeibler(grid,w=data)
penLower = QuadraticLowerBound(grid,x0=0,lb=0)
setting = Setting(op=conv,penalty=penLower,data_fid=KL,regpar=alpha)
print(setting.is_convex)
print(setting.is_tikhonov)
print(setting.primal_setting)
setting.display_all_methods()

The minimization problem is quite difficult to solve: As the gradient of the data fidelity term is singular at 0, gradient-based methods such as FISTA and FB are unstable, and overall there is no linearly convergent minimization method available. 

It is common to practice to use a shifted version of KL, i.e. replace the data fidelity term by
$$
\mathrm{KL}(g^{\mathrm{obs}}+\sigma,Tf+\sigma)
$$
This may account for background noise in the data. (In the case a shift of the data is not necessary.) 

In [ ]:
sigma = 1.

KL_shift = KullbackLeibler(grid,w=sigma*grid.ones(),data=data).shift(-sigma)

set_shift = Setting(op=conv,penalty=penLower,data_fid=KL_shift,regpar=alpha)

set_shift.display_all_methods()

So far, nothing changes as the methods don't "know" that negative values are not attainable. Therefore, we change the data fidelity functional on the negative half-axis to a more favorable behavior using a first (or second) order Taylor approximation at 0. This does not change the optimization problem, but allows us to use an accelerated version of FISTA. Moreover, the dual problem does change in a favorable way. 

In [ ]:

KL_shift_lin  = KullbackLeibler(grid,sigma*grid.ones(), data=data, lin_taylor_l=sigma).shift(np.broadcast_to(-sigma,grid.shape))
#KL_shift_quad = KullbackLeibler(grid,w=data+sigma, quad_taylor_l=sigma).shift(np.broadcast_to(-sigma,grid.shape))

set_KL_shift_lin  = Setting(op=conv, penalty=penLower, data_fid = KL_shift_lin, regpar=alpha)
#set_KL_shift_quad = Setting(op=conv, penalty=penLower, data_fid = KL_shift_quad,regpar=alpha)

set_KL_shift_lin.evaluate_methods()
set_KL_shift_lin.display_all_methods()

#set_KL_shift_quad.evaluate_methods()
#set_KL_shift_quad.display_all_methods()

In [ ]:
methods =  ['FISTA','dual_FISTA','PDHG','dual_PDHG','dual_FB']
print(set_KL_shift_lin._methods)
recos = {}
for method in methods:
    recos[method]= set_KL_shift_lin.run(method)[0]

In [ ]:
convergence_plot(set_KL_shift_lin,methods,'Shifted KL lin, L^2 nonnegativity constraint')
set_KL_shift_lin._methods['dual_FISTA']['stoprule'].history_dict["duality gap"]

In [ ]:
plot_recos(set_KL_shift_lin,methods,recos,truth=exact_sol)

## Kullback-Leibler data fidelity with relative entropy penalty

Let as now replace the penalty term by the relative entropy with respect to some initial guess $f_0$, which 
is given by the Kullback-Leibler divergence as a function of the first argument: 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\frac{1}{\alpha}\mathrm{KL}(g^{\mathrm{obs}}+\sigma,h*f+\sigma) +  \mathrm{KL}(f,f_0)\right].
$$

In [ ]:
f0 =  200*grid.ones()
alpha = 1e-1
RE = RelativeEntropy(grid,w =f0)
set_KL_RE = Setting(op=conv, penalty=RE, data_fid = KL_shift_lin,regpar=alpha)

set_KL_RE.display_all_methods()

In [ ]:
methods =  ['FB','FISTA','PDHG','dual_PDHG','ADMM']
recos = {}
for method in methods:
    recos[method]= set_KL_RE.run(method)[0]

FB and FISTA are both unstable here, ending up in NANs

In [ ]:
convergence_plot(set_KL_RE,methods,'Shifted KL lin, relative entropy')

In [ ]:
plot_recos(set_KL_RE,methods,recos,truth=exact_sol)

We can significantly improve performance by incorporating an upper bound into the penalty term. This make the penalty term strongly convex and its conjugate has a Lipschitz continuous gradient. This Ii particular yields dual_FISTA applicable again. 

In [ ]:
RE_ub = RelativeEntropy(grid,w = f0,constr_u=400)
set_KL_RE_ub = Setting(op=conv, penalty=RE_ub, data_fid = KL_shift_lin,regpar=alpha)

set_KL_RE_ub.display_all_methods()

In [ ]:
methods =  ['FISTA','dual_FISTA','PDHG','dual_PDHG','ADMM']
recos = {}
for method in methods:
    recos[method]= set_KL_RE_ub.run(method)[0]

In [ ]:
convergence_plot(set_KL_RE_ub,methods,'Shifted KL, relative entropy with upper bound')

In [ ]:
plot_recos(set_KL_RE_ub,methods,recos,truth=exact_sol)